Perceptron with Cross Validation - uses MNIST

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import umap  # pip install umap-learn is the installation command
import numpy as np
from sklearn.datasets import fetch_openml


TEST / TRAIN split. No cross-validation.
I had to be very careful in copying my sample code to change X to X-train; this caused subtle errors where the clustering was clearly not working right but the code was running

In [3]:

# mnist as a numpy array
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
X = X[:50000] / 255.0  # Normalize pixels
y = y[:50000]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=0)

# reduce dimensions with  umap - this takes 47 seconds
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1)
X_red_train = reducer.fit_transform(X_train)
X_red_test = reducer.transform(X_test)
from sklearn.linear_model import Perceptron
ppn = Perceptron(eta0=0.1, random_state=1)
ppn.fit(X_red_train, y_train) # train the algorithm


Perceptron(eta0=0.1, random_state=1)

In [4]:
print(y_test[:10])  # print the first 10 test labels. They are strings, not numbers.

['2' '2' '5' '5' '9' '7' '2' '7' '3' '4']


In [5]:
# use the test data
y_pred = ppn.predict(X_red_test)
print('Misclassified examples: %d' % (y_test != y_pred).sum())

Misclassified examples: 8181


Define your own accuracy function.  

In [8]:
def calculate_accuracy(y, y_pred):
  counter = 0
  for i in range(len(y)):
    if y[i] == y_pred[i]:
      counter += 1
  return counter / len(y)

test  with toy example

In [9]:
def test_accuracy():
    y_pred = [1,0,0,0]
    y = [0,0,0,1]
    accuracy = calculate_accuracy(y, y_pred)
    print(f"Should be 0.50: {accuracy:.4f}")
test_accuracy()

Should be 0.50: 0.5000


In [10]:
calculate_accuracy(y_test, y_pred) # call your function

0.59095

Here's the 'real' way to do it:

In [12]:
print('Train set accuracy: %.3f' % ppn.score(X_red_train, y_train))  #about 52% accurate
# do the same for the test set - YOUR CODE HERE
print('Test set accuracy" %.3f' % ppn.score(X_red_test, y_test))

Train set accuracy: 0.597
Test set accuracy" 0.591


# now try cross-validation

In [16]:
from sklearn.model_selection import StratifiedKFold
kfold = StratifiedKFold(n_splits=5).split(X_train, y_train)
scores = []
for k, (train, test) in enumerate(kfold):
   ppn.fit(X_train[train], y_train[train])
   score = ppn.score(X_train[test], y_train[test])
   scores.append(score)
   print(f'Fold: {k+1:02d}, 'f'Acc.: {score:.3f}')

Fold: 01, Acc.: 0.885
Fold: 02, Acc.: 0.845
Fold: 03, Acc.: 0.885
Fold: 04, Acc.: 0.889
Fold: 05, Acc.: 0.880


In [14]:
mean_acc = np.mean(scores)
std_acc = np.std(scores)
print(f'\n Cross Validation accuracy: {mean_acc:.3f} +/- {std_acc:.3f}')


 Cross Validation accuracy: 0.877 +/- 0.016


This is fine, but if we want to try changing a parameter (like the number of folds), then we need to do train/validation/test split not just test/train split.
Since we didn't actually use our entire MNIST dataset, we have effectively 'held out' some data and the test & train data is separate. So, we could optimize a parameter based on the method above and then when we are all done, get the final accuracy of the best choice by testing the trained perceptron on the unused data.

Automate (using a loop) the calculation of the (mean) CV accuracy - loop over choices of K, which is the number of folds. Select the best value of K and then get a final accuracy value using the unused data.

In [18]:
from sklearn.model_selection import StratifiedKFold


for i in range(6, 15, 2):
  kfold = StratifiedKFold(n_splits=i).split(X_train, y_train)
  scores = []
  for k, (train, test) in enumerate(kfold):
    ppn.fit(X_train[train], y_train[train])
    score = ppn.score(X_train[test], y_train[test])
    scores.append(score)
    print(f'Fold: {k+1:02d}, 'f'Acc.: {score:.3f}')
  mean_acc = np.mean(scores)
  std_acc = np.std(scores)
  print(f'\n Cross Validation accuracy: {mean_acc:.3f} +/- {std_acc:.3f}')


Fold: 01, Acc.: 0.842
Fold: 02, Acc.: 0.863
Fold: 03, Acc.: 0.875
Fold: 04, Acc.: 0.879
Fold: 05, Acc.: 0.865
Fold: 06, Acc.: 0.875

 Cross Validation accuracy: 0.866 +/- 0.012
Fold: 01, Acc.: 0.829
Fold: 02, Acc.: 0.863
Fold: 03, Acc.: 0.863
Fold: 04, Acc.: 0.850
Fold: 05, Acc.: 0.882
Fold: 06, Acc.: 0.857
Fold: 07, Acc.: 0.887
Fold: 08, Acc.: 0.878

 Cross Validation accuracy: 0.864 +/- 0.018
Fold: 01, Acc.: 0.890
Fold: 02, Acc.: 0.854
Fold: 03, Acc.: 0.868
Fold: 04, Acc.: 0.882
Fold: 05, Acc.: 0.876
Fold: 06, Acc.: 0.865
Fold: 07, Acc.: 0.821
Fold: 08, Acc.: 0.878
Fold: 09, Acc.: 0.896
Fold: 10, Acc.: 0.840

 Cross Validation accuracy: 0.867 +/- 0.022
Fold: 01, Acc.: 0.867
Fold: 02, Acc.: 0.856
Fold: 03, Acc.: 0.872
Fold: 04, Acc.: 0.869
Fold: 05, Acc.: 0.849
Fold: 06, Acc.: 0.868
Fold: 07, Acc.: 0.870
Fold: 08, Acc.: 0.854
Fold: 09, Acc.: 0.874
Fold: 10, Acc.: 0.861
Fold: 11, Acc.: 0.867
Fold: 12, Acc.: 0.877

 Cross Validation accuracy: 0.865 +/- 0.008
Fold: 01, Acc.: 0.872
Fold: 

Based on my values (6, 8, 10, 12, 14), k = 14 is the best accuracy. It has a cross validation accuracy of 0.872 +/- 0.011 or about 87.2%

In [20]:
kfold = StratifiedKFold(n_splits=14).split(X_test, y_test)
scores = []
for k, (train, test) in enumerate(kfold):
  ppn.fit(X_test[train], y_test[train])
  score = ppn.score(X_test[test], y_test[test])
  scores.append(score)
  print(f'Fold: {k+1:02d}, 'f'Acc.: {score:.3f}')
mean_acc = np.mean(scores)
std_acc = np.std(scores)
print(f'\n Cross Validation accuracy: {mean_acc:.3f} +/- {std_acc:.3f}')

Fold: 01, Acc.: 0.886
Fold: 02, Acc.: 0.886
Fold: 03, Acc.: 0.882
Fold: 04, Acc.: 0.885
Fold: 05, Acc.: 0.861
Fold: 06, Acc.: 0.863
Fold: 07, Acc.: 0.836
Fold: 08, Acc.: 0.808
Fold: 09, Acc.: 0.871
Fold: 10, Acc.: 0.889
Fold: 11, Acc.: 0.873
Fold: 12, Acc.: 0.825
Fold: 13, Acc.: 0.854
Fold: 14, Acc.: 0.835

 Cross Validation accuracy: 0.861 +/- 0.025


Final Accuracy: Cross Validation accuracy: 0.861 +/- 0.025